In [1]:
import sys
import importlib
from pathlib import Path

import numpy as np

# Find the repository root robustly from the notebook working directory
cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "utilities" / "functions.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing utilities/functions.py")

sys.path.append(str(repo_root))  # make the repo importable from notebooks in subfolders

import utilities.functions as functions
import utilities.plot as plot

# Reload modules to reflect any changes
importlib.reload(functions)
importlib.reload(plot)

IN_start_index = 1
IN_end_index = 263
PR_start_index = 1
PR_end_index = 99
RT_start_index = 39
RT_end_index = 226

data_root = repo_root / "ms0_5"

IN_seq_path = str(data_root / "IN" / "data" / "in.reduce4.seq")
PR_seq_path = str(data_root / "PR" / "data" / "pr.exper.reduce4.seq")
RT_seq_path = str(data_root / "RT" / "data" / "rt.reduce4.seq")

IN_all_seq = functions.read_seq(IN_seq_path)
PR_all_seq = functions.read_seq(PR_seq_path)
RT_all_seq = functions.read_seq(RT_seq_path)

IN_consensus = str(data_root / "IN" / "data" / "in.consensus.reduce4.seq")
with open(IN_consensus, "r") as f:
    IN_consensus_seq = f.read().strip()

PR_consensus = str(data_root / "PR" / "data" / "pr.consensus.reduce4.seq")
with open(PR_consensus, "r") as f:
    PR_consensus_seq = f.read().strip()

RT_consensus = str(data_root / "RT" / "data" / "rt.consensus.reduce4.seq")
with open(RT_consensus, "r") as f:
    RT_consensus_seq = f.read().strip()

IN_redux = functions.get_redu_dict(str(data_root / "IN" / "data" / "in.reduce4.redux"), 1)
PR_redux = functions.get_redu_dict(str(data_root / "PR" / "data" / "pr.reduce4.redux"), 0)
RT_redux = functions.get_redu_dict(str(data_root / "RT" / "data" / "rt.reduce4.redux"), 0)

alphabet = ["A", "B", "C", "D"]


def build_J_matrix(j_file, min_position, max_position):
    """Dense coupling tensor, shape (L, L, 4, 5), indexed [p1, p2, aa_at_p1, aa_at_p2].

    Same content as functions.load_J_dict but as an array instead of a ~1M-entry
    dict, so couplings can be gathered with numpy instead of per-key lookups.
    The 5th slot on the last axis is an all-zero column standing in for
    out-of-alphabet characters (mirrors the J_dict.get(..., 0) default).
    """
    J = np.load(j_file).astype(np.float32)
    L = max_position - min_position + 1
    Jm = np.zeros((L, L, 4, 5), dtype=np.float32)
    # load_J_dict walks pos1 ascending, pos2 from pos1+1 ascending -> row-major upper triangle
    iu0, iu1 = np.triu_indices(L, 1)
    blocks = J.reshape(-1, 4, 4)
    assert blocks.shape[0] == iu0.size, "J.npy row count does not match the position range"
    Jm[iu0, iu1, :, :4] = blocks
    Jm[iu1, iu0, :, :4] = blocks.transpose(0, 2, 1)
    return Jm


IN_J = build_J_matrix(str(data_root / "IN" / "data" / "J.npy"), IN_start_index, IN_end_index)
PR_J = build_J_matrix(str(data_root / "PR" / "data" / "J_PR.npy"), PR_start_index, PR_end_index)
RT_J = build_J_matrix(str(data_root / "RT" / "data" / "J_RT.npy"), RT_start_index, RT_end_index)

IN_all_seq_unreduced = functions.read_seq(str(data_root / "IN" / "data" / "in.fullseq"))
PR_all_seq_unreduced = functions.read_seq(str(data_root / "PR" / "data" / "pr.exper.fullseq"))
RT_all_seq_unreduced = functions.read_seq(str(data_root / "RT" / "data" / "rt.fullseq"))

In [2]:
import csv

import numpy as np

# ---------------------------------------------------------------------------
# Vectorized re-implementation of the per-sequence loop.
#
# For a pair (p1: wt1->mt1, p2: wt2->mt2), calculate_dde_v2 first rewrites the
# sequence to wild type at both positions, so every quantity depends on the
# sequence only through the "background" couplings
#
#     S(p, x) = sum_{o not in {p1, p2}} J[p, o, x, seq[o]]
#
# plus the direct term J[p1, p2, ., .]. With
#     M[a1, a2] = S(p1, a1) + S(p2, a2) + J[p1, p2, a1, a2]
# we get exactly what the scalar functions compute:
#     dE1  = M[wt1, wt2] - M[mt1, wt2]
#     dE2  = M[wt1, wt2] - M[wt1, mt2]
#     dE12 = M[wt1, wt2] - M[mt1, mt2]
#     p_DMC = exp(dE12) / sum_{a1,a2} exp(M[wt1, wt2] - M[a1, a2])   (a softmax over -M)
# so the 8 numbers S(p1, ABCD) / S(p2, ABCD) are all that is needed per sequence,
# and they come from a single matmul against a one-hot encoding of the alignment.
#
# Gain of fitness is the position-local test -- dE12 beats the wild type and both of the
# pair's own single mutants -- plus the contender test: the pair also has to be the
# strongest of the double mutations it is in *conflict* with. Two double mutations are in
# conflict when one takes a position of the other with a *different* amino acid, since then
# they can never hold in the same sequence and only one of them can own it. Sharing a
# position with the *same* residue is not a conflict -- M41L-T215F, M41L-D67N and D67N-T215F
# can all hold at once -- while M41L-T215F and M41L-T215Y can not. This is the definition
# used in contending_dm_pairs_v1.ipynb.
#
# The contenders of this pair are therefore every (p: u_p -> a, q: u_q -> b) with p one of
# its two positions, a any residue other than u_p and other than the pair's own mutant
# there, q *any* other position in the protein and b a mutation at q. Not just the 16
# residue combinations at (p1, p2): those are only the contenders whose partner position
# happens to be the other one of the pair, together with the wild type and the single
# mutants (which the local test already covers) and with competing *single* mutations at
# p1 or p2, which are not double mutations and so are not contenders at all. That full
# range does not factorize through S; it is built instead from the full backgrounds
#
#     T(p, x) = sum_{o != p} J[p, o, x, seq[o]]
#
# (one matmul for the whole alignment) with the partner position subtracted back out --
# see _best_competing_dmc.
#
# The contender half of the rule is switchable: NON_OVERLAPPING (set in the flags cell) keeps
# it, NON_OVERLAPPING = False drops it and leaves gain of fitness as the position-local test
# alone -- dE12 > dE1, dE12 > dE2, dE12 > 0 -- so a sequence can then be gain of fitness for
# several mutually exclusive double mutations at once. Everything built from T is skipped in
# that mode, which is also where nearly all of the run time goes.
# ---------------------------------------------------------------------------

_AA_CODE = np.full(256, 4, dtype=np.uint8)  # anything outside ABCD -> the zero-coupling slot
for _i, _c in enumerate("ABCD"):
    _AA_CODE[ord(_c)] = _i


def encode_seqs(seq_list, min_pos, max_pos):
    """One-hot encode an alignment as (N, L*5) float32, column order (position, aa)."""
    L = max_pos - min_pos + 1
    N = len(seq_list)
    raw = np.frombuffer("".join(seq_list).encode(), dtype=np.uint8)
    assert raw.size == N * L, "all sequences must span exactly min_pos..max_pos"
    codes = _AA_CODE[raw.reshape(N, L)]
    onehot = np.zeros((N * L, 5), dtype=np.float32)
    onehot[np.arange(N * L), codes.ravel()] = 1.0
    return codes, onehot.reshape(N, L * 5)


def _site_energies(onehot, Jm, p, excluded):
    """S(p, x) for x in ABCD, for every sequence -> (N, 4)."""
    A = Jm[p].copy()
    A[list(excluded)] = 0.0  # drop the two mutated positions from the background sum
    return np.asarray(onehot @ A.transpose(0, 2, 1).reshape(-1, 4), dtype=np.float64)


def pair_energies(onehot, Jm, p1i, p2i, wt1, mt1, wt2, mt2):
    """dE1, dE2, dE12 and p(DMC) of one pair, per sequence.

    Only the pair's own energies: dE12 against the wild type and against its own two single
    mutants. Every comparison against a *competing* mutation is the contender test, which
    lives in _best_competing_dmc -- the other residue combinations at (p1, p2) are not
    tested here, because the ones that matter (the genuine double mutations among them) are
    contenders like any other and are covered there, with q = the partner position.
    """
    S1 = _site_energies(onehot, Jm, p1i, (p1i, p2i))
    S2 = _site_energies(onehot, Jm, p2i, (p1i, p2i))
    J12 = Jm[p1i, p2i, :, :4].astype(np.float64)

    M = S1[:, :, None] + S2[:, None, :] + J12[None, :, :]
    base = M[:, wt1, wt2]
    de1 = base - M[:, mt1, wt2]
    de2 = base - M[:, wt1, mt2]
    de12 = base - M[:, mt1, mt2]

    # softmax over -M (the +base cancels); done shifted, so no exp overflow
    Z = -M.reshape(S1.shape[0], 16)
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    p_dmc = E[:, mt1 * 4 + mt2] / E.sum(axis=1)
    return de1, de2, de12, p_dmc


def _background_energies(onehot, Jm):
    """T[n, p, x] = sum_{o != p} J[p, o, x, seq[o]], shape (N, L, 4).

    One matmul for the whole alignment; the (p, p) block of Jm is zero, so it drops out
    of the sum on its own. Everything built on top of T is a difference of large sums, so
    this is accumulated in float64.
    """
    L = Jm.shape[0]
    W = np.asarray(Jm, dtype=np.float64).transpose(0, 2, 1, 3).reshape(L * 4, L * 5)
    return (np.asarray(onehot, dtype=np.float64) @ W.T).reshape(-1, L, 4)


def _de12_from_T(T, Jm, codes, p, q, a, b, u_p, u_q):
    """dE12 of one double mutation (p: u_p -> a, q: u_q -> b), per sequence.

    Same quantity as pair_energies' de12, expressed through T instead of S: each
    background has to give back the coupling to the other mutated position, and the
    direct p-q coupling is added on top.
    """
    sp, sq = codes[:, p], codes[:, q]
    Jpq = np.asarray(Jm[p, q], dtype=np.float64)   # (4, 5), J[p, q, aa_at_p, aa_at_q]
    Jqp = np.asarray(Jm[q, p], dtype=np.float64)   # (4, 5), J[q, p, aa_at_q, aa_at_p]
    return ((T[:, p, u_p] - T[:, p, a]) + (T[:, q, u_q] - T[:, q, b])
            - (Jpq[u_p, sq] - Jpq[a, sq]) - (Jqp[u_q, sp] - Jqp[b, sp])
            + (Jpq[u_p, u_q] - Jpq[a, b]))


def _best_competing_dmc(T, Jm, codes, p, u_p, m_p, u, chunk=2048):
    """Strongest double mutation in conflict with a pair that mutates p to m_p, per sequence.

    Ranges over every (p: u_p -> a, q: u_q -> b) with q any other position in the protein,
    b != u_q, and a outside {u_p, m_p}: a different amino acid at the shared position p is
    what makes two double mutations mutually exclusive, and one shared position is enough --
    the partner position q is unconstrained, so it covers both a contender that takes the
    pair's other position (q = p2, e.g. M41L-T215F vs M41V-T215Y) and one that goes off to a
    third position (M41L-T215F vs M41V-D67N). a == m_p is skipped because those agree with the
    pair at p and can occur together (M41L-T215F and M41L-D67N), and it is also what keeps the
    calling pair out of its own comparison. b != u_q keeps the range to genuine double
    mutations: a single mutation is not a contender. Wild type is u_p at p (the pair's own
    labelled wild type) and the consensus residue u[q] at the partner position.

    Returns the best dE12 per sequence. Sequences are processed in chunks because the
    intermediate is (chunk, L, 4, 4).
    """
    N, L, _ = T.shape
    qs = np.arange(L)
    aa = np.arange(4)
    Jp = np.asarray(Jm[p], dtype=np.float64)     # (L, 4, 5), J[p, q, a, aa_at_q]
    Jq = np.asarray(Jm[:, p], dtype=np.float64)  # (L, 4, 5), J[q, p, b, aa_at_p]
    direct = Jp[qs, u_p, u][:, None, None] - Jp[:, :, :4]   # (L, 4, 4)

    invalid = np.zeros((L, 4, 4), dtype=bool)
    invalid[p] = True             # q must be a different position
    invalid[:, u_p, :] = True     # a must actually be a mutation at p
    invalid[:, m_p, :] = True     # a == m_p agrees with the pair at p -> compatible, not a rival
    invalid[qs, :, u] = True      # b must actually be a mutation at q
    invalid = invalid.reshape(L * 16)

    best = np.empty(N)
    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        cb, Ts = codes[s:e], T[s:e]
        sp = cb[:, p]
        dp = Ts[:, p, u_p][:, None] - Ts[:, p, :]                          # (B, 4)
        dq = np.take_along_axis(Ts, u[None, :, None], axis=2) - Ts         # (B, L, 4)
        cA = (Jp[qs[None, :], u_p, cb][:, :, None]
              - Jp[qs[None, :, None], aa[None, None, :], cb[:, :, None]])  # (B, L, 4)
        cB = (Jq[qs, u, sp[:, None]][:, :, None]
              - Jq[qs[None, :, None], aa[None, None, :], sp[:, None, None]])  # (B, L, 4)
        dE = (dp[:, None, :, None] + dq[:, :, None, :]
              - cA[:, :, :, None] - cB[:, :, None, :] + direct[None])
        flat = dE.reshape(e - s, L * 16)
        flat[:, invalid] = -np.inf
        best[s:e] = flat.max(axis=1)
    return best


def _cat_stats(mask, p, weights, dmc, len_all_seqs, weights_sum):
    """Counts / average p / observed f for one epistasis subcategory."""
    n = int(mask.sum())
    n_dmc = int(np.count_nonzero(mask & dmc))
    p_sum = float(p[mask].sum())
    w_sub = float(weights[mask].sum())
    wp_sum = float(np.dot(p[mask], weights[mask]))
    w_dmc = float(weights[mask & dmc].sum())
    return {
        'n': n,
        'n_dmc': n_dmc,
        'w_sum': w_sub,
        'w_dmc': w_dmc,
        'avg_p_total': p_sum / len_all_seqs if len_all_seqs else 0,
        'avg_p_sub': p_sum / n if n else 0,
        'w_p_total': wp_sum / weights_sum if n else 0,
        'w_p_sub': wp_sum / w_sub if n else 0,
        'obs_f_total': n_dmc / len_all_seqs,
        'obs_f_sub': n_dmc / n if n else 0,
        'w_obs_f_total': w_dmc / weights_sum if n else 0,
        'w_obs_f_sub': w_dmc / w_sub if n else 0,
    }


def output_probs(prefix, min_pos, max_pos, all_seq, consensus_seq, redux, pairs, weights_path, J, output_csv):
    """
    Process epistasis data for a given prefix (e.g., IN, PR, RT).

    A sequence counts as gain of fitness for a pair when the pair's dE12 beats the wild type
    and beats both of its own single mutants; with NON_OVERLAPPING on it must also beat every
    double mutation it is in conflict with. Two double mutations are in conflict when one takes
    a position of the other with a *different* amino acid -- one shared position with a
    different residue is enough, and the partner position can be anywhere in the protein, not
    just among the pairs listed in `pairs`. Conflicting double mutations cannot happen in the
    same sequence, so the pairs a sequence is gain of fitness for always agree on every position
    they share: M41L-T215F, M41L-D67N and D67N-T215F can all hold at once, but M41L-T215F and
    M41L-T215Y never can. Pairs that clear the wild-type/singles bar but lose to a contender
    fall through to rescue.

    With NON_OVERLAPPING off that last requirement is dropped: gain of fitness is the local test
    alone, and the same sequence can then be gain of fitness for double mutations that exclude
    each other -- which is the double counting the contender rule exists to remove.

    Parameters:
        prefix (str): Prefix for the dataset (e.g., 'IN', 'PR', 'RT').
        all_seq (list): List of all sequences.
        consensus_seq (str): Consensus sequence.
        redux (dict): Reduction dictionary.
        pairs (list): List of mutation pairs.
        weights_path (str): Path to the weights file.
        J (np.ndarray): Coupling tensor from build_J_matrix.
        output_csv (str): Output CSV file name.
    """
    len_all_seqs = len(all_seq)

    # Read weights from the file
    with open(weights_path, 'r') as f:
        weights = [float(line.strip()) for line in f]

    # Ensure the weights list matches the all_seq list
    assert len(weights) == len(all_seq), "Weights and sequences must have the same length."

    weights = np.asarray(weights, dtype=np.float64)
    weights_sum = float(weights.sum())

    # One-hot encoding and backgrounds are shared by every pair -> build them once
    codes, onehot = encode_seqs(all_seq, min_pos, max_pos)
    consensus_codes, consensus_onehot = encode_seqs([consensus_seq], min_pos, max_pos)
    u = consensus_codes[0]                      # wild type at every position
    # the contender machinery is only built when the rule is on: T and the scans are where
    # essentially all of the run time and memory of this function go
    T = _background_energies(onehot, J) if NON_OVERLAPPING else None
    competing = {}                              # (position, wild type, mutant) -> best contender
    print("  gain of fitness: local test"
          + (" + no contender may beat it (NON_OVERLAPPING)" if NON_OVERLAPPING
             else " only -- contender rule off (NON_OVERLAPPING = False)"))

    csv_total_data = []
    csv_data = []
    csv_weighted_data = []

    for pair in pairs:
        print(f"Processing pair: {pair}")
        pair1, pair2 = functions.split_pairs(pair)
        p1_reduced = functions.unreduced_to_reduced(redux, pair1)
        p2_reduced = functions.unreduced_to_reduced(redux, pair2)
        wt1, pos1, mt1 = functions.split_pair(p1_reduced)
        wt2, pos2, mt2 = functions.split_pair(p2_reduced)

        p1i, p2i = pos1 - min_pos, pos2 - min_pos
        iwt1, imt1 = "ABCD".index(wt1), "ABCD".index(mt1)
        iwt2, imt2 = "ABCD".index(wt2), "ABCD".index(mt2)
        idx = (p1i, p2i, iwt1, imt1, iwt2, imt2)

        # The contender scan takes the wild type at the *partner* position from the consensus,
        # so a pair whose own labelled wild type is not the consensus does not sit in the same
        # reference state as its contenders, and its contender set can be short a few rivals.
        if NON_OVERLAPPING and (u[p1i] != iwt1 or u[p2i] != iwt2):
            print(f"  warning: {pair} is labelled {wt1}{pos1}{mt1}-{wt2}{pos2}{mt2} but the "
                  f"consensus is {'ABCD'[u[p1i]]} at {pos1} and {'ABCD'[u[p2i]]} at {pos2}; "
                  f"its contender set is enumerated against the consensus")

        # consensus reference for the flip test
        c_de1, c_de2, _, _ = pair_energies(consensus_onehot, J, *idx)
        consensus_de_diff = float(c_de1[0] - c_de2[0])

        de1, de2, de12, p_SH = pair_energies(onehot, J, *idx)

        # sequences carrying the double mutant combination
        dmc = (codes[:, p1i] == imt1) & (codes[:, p2i] == imt2)

        # ---- strongest conflicting double mutation at each of the two positions ----
        # cached per (position, wild type, mutant): pairs with the same mutation share the scan
        if NON_OVERLAPPING:
            for key in ((p1i, iwt1, imt1), (p2i, iwt2, imt2)):
                if key not in competing:
                    competing[key] = _best_competing_dmc(T, J, codes, key[0], key[1], key[2], u)
            own = _de12_from_T(T, J, codes, p1i, p2i, imt1, imt2, iwt1, iwt2)
            no_rival = ((own > competing[(p1i, iwt1, imt1)])
                        & (own > competing[(p2i, iwt2, imt2)]))
        else:
            no_rival = np.ones(len_all_seqs, dtype=bool)   # rule off: nothing to beat

        # ---- epistasis subcategories (same branch order as the scalar version) ----
        de_diff = de1 - de2
        flip = consensus_de_diff * de_diff < 0
        non_flip = consensus_de_diff * de_diff > 0

        both_below = (de1 < de12) & (de2 < de12)
        # gain of fitness: better than the wild type and than both of the pair's own single
        # mutants, and -- with NON_OVERLAPPING on -- than every double mutation in conflict with
        # it: one that takes one of its two positions with a different amino acid, with any
        # partner position in the protein (no_rival, all True when the flag is off). Sequences
        # that clear the wild-type/singles bar but lose to a contender fall through to rescue,
        # as the elif chain dictates.
        gof = both_below & (de12 > 0) & no_rival
        rescue = both_below & ~gof
        comp = ~both_below & ((de1 < de12) | (de2 < de12))
        noncomp = ~both_below & ~((de1 < de12) | (de2 < de12))

        stats = {name: _cat_stats(mask, p_SH, weights, dmc, len_all_seqs, weights_sum)
                 for name, mask in (('gof', gof), ('rescue', rescue), ('comp', comp),
                                    ('noncomp', noncomp), ('flip', flip), ('non_flip', non_flip))}

        # ---- totals over all sequences ----
        total_counts = len_all_seqs
        total_with_DMC_count = int(np.count_nonzero(dmc))
        average_p_all = float(p_SH.sum()) / len_all_seqs if len_all_seqs else 0
        weighted_all_prob = float(np.dot(p_SH, weights)) / weights_sum if len_all_seqs else 0
        weights_with_DMC_sum = float(weights[dmc].sum())
        actual_p_all = total_with_DMC_count / total_counts if total_counts > 0 else 0
        actual_p_all_weighted = weights_with_DMC_sum / weights_sum if len_all_seqs else 0

        # Append data
        csv_total_data.append([pair, 'Total', total_counts, total_with_DMC_count, average_p_all, actual_p_all, average_p_all, actual_p_all, weights_sum, weights_with_DMC_sum, weighted_all_prob, actual_p_all_weighted, weighted_all_prob, actual_p_all_weighted])

        # Append data for each epistasis subset
        # non-weighted
        csv_data.append(['total'])
        for label, key in (('Gain_of_function', 'gof'), ('rescue', 'rescue'),
                           ('compensatory', 'comp'), ('non_compensatory', 'noncomp')):
            s = stats[key]
            csv_data.append([pair, label, s['n'], s['n_dmc'], s['avg_p_total'], s['obs_f_total'], s['avg_p_sub'], s['obs_f_sub']])
        csv_data.append([])
        for label, key in (('flipped', 'flip'), ('non_flipped', 'non_flip')):
            s = stats[key]
            csv_data.append([pair, label, s['n'], s['n_dmc'], s['avg_p_total'], s['obs_f_total'], s['avg_p_sub'], s['obs_f_sub']])
        csv_data.append([])

        # weighted
        csv_weighted_data.append(['total'])
        for label, key in (('Gain_of_function', 'gof'), ('rescue', 'rescue'),
                           ('compensatory', 'comp'), ('non_compensatory', 'noncomp')):
            s = stats[key]
            csv_weighted_data.append([pair, label, s['w_sum'], s['w_dmc'], s['w_p_total'], s['w_obs_f_total'], s['w_p_sub'], s['w_obs_f_sub']])
        csv_weighted_data.append([])
        s = stats['flip']
        csv_weighted_data.append([pair, 'flipped', s['w_sum'], s['w_dmc'], s['w_p_total'], s['w_obs_f_total'], s['w_p_sub'], s['w_obs_f_sub']])
        s = stats['non_flip']
        # NOTE: last column repeats w_p_sub, as in the original implementation
        csv_weighted_data.append([pair, 'non_flipped', s['w_sum'], s['w_dmc'], s['w_p_total'], s['w_obs_f_total'], s['w_p_sub'], s['w_p_sub']])
        csv_weighted_data.append([])

    with open(output_csv, 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(['mutation_pair', 'epistasis_subset', 'num_seqs', 'num_with_DMC','average_p_total', 'observed_f_total','average_p_subcategory', 'observed_f_subcategory','', 'weights_sum','weights_sum_with_DMC','weighted_average_p_total', 'weighted_observed_f_total', 'weighted_average_p_subcategory', 'weighted_observed_f_subcategory'])
        count = 0
        for row, weighted_row in zip(csv_data, csv_weighted_data):
            if row == [] or weighted_row == []:
                csv_writer.writerow([])
            elif row == ['total'] or weighted_row == ['total']:
                csv_writer.writerow([f"{csv_total_data[count][0]}", f"{csv_total_data[count][1]}", f"{csv_total_data[count][2]}", f"{csv_total_data[count][3]:}", f"{csv_total_data[count][4]:.4f}", f"{csv_total_data[count][5]:.4f}", f"{csv_total_data[count][6]:.4f}",f"{csv_total_data[count][7]:.4f}", '', f"{csv_total_data[count][8]:.4f}", f"{csv_total_data[count][9]:.4f}", f"{csv_total_data[count][10]:.4f}", f"{csv_total_data[count][11]:.4f}", f"{csv_total_data[count][12]:.4f}", f"{csv_total_data[count][13]:.4f}"])
                count += 1
            else:
                csv_writer.writerow([f"{row[0]}", f"{row[1]}", f"{row[2]}", f"{row[3]}", f"{row[4]:.4f}", f"{row[5]:.4f}", f"{row[6]:.4f}", f"{row[7]:.4f}", '', f"{weighted_row[2]:.4f}", f"{weighted_row[3]:.4f}", f"{weighted_row[4]:.4f}", f"{weighted_row[5]:.4f}", f"{weighted_row[6]:.4f}", f"{weighted_row[7]:.4f}"])
    print(f"CSV file {output_csv} created successfully.")


In [3]:
import csv

# =============================================================================
# What counts as gain of fitness
# -----------------------------------------------------------------------------
# NON_OVERLAPPING = True  -> the contender rule (current behaviour): on top of beating the
#     wild type and both of its own single mutants, the pair has to beat every double
#     mutation it is in conflict with -- one that takes one of its two positions with a
#     *different* amino acid, partner position anywhere in the protein. Conflicting double
#     mutations cannot hold in the same sequence, so no sequence is counted as gain of
#     fitness for two of them (the definition in contending_dm_pairs_v1.ipynb).
#
# NON_OVERLAPPING = False -> drop that requirement. Gain of fitness is then purely the
#     position-local test: dE12 > dE1, dE12 > dE2 and dE12 > 0 (better than the wild type).
#     The same sequence can then be gain of fitness for several mutually exclusive double
#     mutations. This mode also skips the contender scan, which is nearly the whole run time.
# =============================================================================
NON_OVERLAPPING = False

# =============================================================================
# Where the mutation pairs come from
# -----------------------------------------------------------------------------
# USE_TOP_DDE_PAIRS = False -> the hand-picked pair lists written out in the cells
#     below (current behaviour): IN, PR, and RT split into NRTI / NNRTI.
#
# USE_TOP_DDE_PAIRS = True  -> the ms0_5 top-1000 ddE pairs instead, keeping only
#     pairs that are actually carried by a decent share of the alignment: the
#     number of sequences carrying *both* mutations (the both_count column of the
#     ms0_5 summary) has to exceed DMC_MIN_PERCENT of the MSA. RT is then run as a
#     whole -- the top-1000 ddE list is not split by drug class, so pooling NRTI and
#     NNRTI into one run is what that pair list means.
#
# both_count > DMC_MIN_PERCENT% is the same filter that produced the pair lists in
# DMC_sebset_freq2.5_with_filter (1% for IN and RT, 0.5% for PR).
# =============================================================================
USE_TOP_DDE_PAIRS = True

# keep a pair when its double-mutation carriers are more than this % of the MSA, per
# protein: IN is a much smaller alignment (1,220 sequences against 5,710 and 19,194), so the
# same cut leaves it with only a handful of pairs
DMC_MIN_PERCENT = {'IN': 1.0, 'PR': 4.0, 'RT': 5.0}
TOP_N_PAIRS = 20        # of the survivors, keep this many highest-ddE pairs; None -> keep all of them

TOP_DDE_FILES = {
    'IN': 'IN_top_1000_dde_summary.csv',
    'PR': 'PR_top_1000_dde_summary.csv',
    # The NRTI and NNRTI files list the same 1000 pairs with the same counts and the
    # same ddE -- they differ only in the DRM flag columns, which are not used here --
    # so either one is the RT-as-a-whole list.
    'RT': 'RT_NRTI_top_1000_dde_summary.csv',
}

_DEFAULT = object()  # "not given" -> fall back to the globals above, so top_n=None can mean "all"


def load_top_dde_pairs(protein, n_seqs, min_percent=_DEFAULT, top_n=_DEFAULT, verbose=True):
    """ms0_5 top-1000 ddE pairs for `protein`, filtered by double-mutation-carrier count.

    A pair is kept when the number of sequences carrying both of its mutations
    (both_count) is more than `min_percent` percent of `n_seqs`, the size of the MSA.
    The summary files are written in descending ddE order and that order is preserved,
    so the first `top_n` survivors are the strongest-ddE ones; top_n=None keeps them all.

    min_percent defaults to this protein's entry in DMC_MIN_PERCENT and top_n to
    TOP_N_PAIRS; pass either explicitly to override it for one call, e.g.
        load_top_dde_pairs('PR', len_PR_all_seqs, min_percent=0.5)
    """
    min_percent = DMC_MIN_PERCENT[protein] if min_percent is _DEFAULT else min_percent
    top_n = TOP_N_PAIRS if top_n is _DEFAULT else top_n

    path = data_root / TOP_DDE_FILES[protein]
    with open(path, newline='') as f:
        rows = list(csv.DictReader(f))

    cutoff = n_seqs * min_percent / 100.0
    kept = [r['double_mutation_pair'] for r in rows if float(r['both_count']) > cutoff]
    n_passing = len(kept)
    if top_n is not None:
        kept = kept[:top_n]

    if verbose:
        print(f"{protein}: {n_passing} of {len(rows)} top-ddE pairs have more than "
              f"{cutoff:.1f} double-mutation carriers ({min_percent}% of {n_seqs} sequences)"
              f" -> using {len(kept)}"
              + (f" (top {top_n} by ddE)" if top_n is not None else ""))
    return kept


def out_name(stem, protein):
    """Output CSV name, tagged with the switches above so that runs made under different
    settings do not overwrite each other's results.

    `protein` picks the threshold that goes in the tag, now that it differs per protein.
    """
    tags = []
    if USE_TOP_DDE_PAIRS:
        tags.append(f"topdde_{DMC_MIN_PERCENT[protein]:g}pct")
        if TOP_N_PAIRS is not None:
            tags.append(f"top{TOP_N_PAIRS}")
    if not NON_OVERLAPPING:
        tags.append("with_overlap")   # gain of fitness without the contender rule
    return "_".join([stem, *tags]) + ".csv"


In [4]:
IN_weights_path = 'IN/data/in.weights.txt'
len_IN_all_seqs = len(IN_all_seq)
IN_weights_path = str(data_root / "IN" / "data" / "in.weights.txt")
with open(IN_weights_path, "r") as f:
    IN_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(IN_weights) == len_IN_all_seqs, "Weights and sequences must have the same length."

IN_pairs = [
    'G140S-Q148H',
    'Y143C-S230R',
    'G140A-Q148K',
    'G140S-Q148R',
    'G140S-Q148K',
    'G140A-Q148R',
    'E138K-Q148K',
    'G140A-Q148H',
    'E138K-Q148R',
    'Y143C-S230K',
    'N155H-E170A',
    'E138K-S147G',
    'S147G-Q148R',
    'Y143R-I151V',
    'E138K-Q148H',
    'S147G-L158V',
    'E92Q-K215R',
    'E138A-Q148H',
    'E138K-S230K',
    'E138K-Y194C',
    # 'G140S-Q148H',
    # 'Y143C-S230R',
    # 'E157Q-K160Q',
    # 'S119G-T122I',
    # 'G140S-Q148R',
    # 'K219N-N222K',
    # 'E138K-Q148R',
    # 'E11D-S195T',
    # 'T125A-V126L',
    # 'S119P-T122I',
    # 'Q221S-N222K',
    # 'L28I-V37I',
    # 'T97A-Y143R',
    # 'E11D-S24N',
    # 'E138K-S147G',
    # 'D6E-E10D',
    # 'T97A-S119R',
    # 'S255N-D256E',
    # 'I220L-Y227F',
    # 'E11D-A21T',
]

if USE_TOP_DDE_PAIRS:
    IN_pairs = load_top_dde_pairs('IN', len_IN_all_seqs)

output_probs('IN', 1,263,IN_all_seq, IN_consensus_seq, IN_redux, IN_pairs, IN_weights_path, IN_J, out_name('integrase_all_probabilities_v17', 'IN'))


IN: 39 of 1000 top-ddE pairs have more than 12.2 double-mutation carriers (1.0% of 1220 sequences) -> using 20 (top 20 by ddE)
  gain of fitness: local test only -- contender rule off (NON_OVERLAPPING = False)
Processing pair: G140S-Q148H
Processing pair: Y143C-S230R
Processing pair: E157Q-K160Q
Processing pair: S119G-T122I
Processing pair: G140S-Q148R
Processing pair: K219N-N222K
Processing pair: E138K-Q148R
Processing pair: E11D-S195T
Processing pair: T125A-V126L
Processing pair: S119P-T122I
Processing pair: Q221S-N222K
Processing pair: L28I-V37I
Processing pair: T97A-Y143R
Processing pair: E11D-S24N
Processing pair: E138K-S147G
Processing pair: D6E-E10D
Processing pair: T97A-S119R
Processing pair: S255N-D256E
Processing pair: I220L-Y227F
Processing pair: E11D-A21T
CSV file integrase_all_probabilities_v17_topdde_1pct_top20_with_overlap.csv created successfully.


In [5]:
PR_weights_path = str(data_root / "PR" / "data" / "pr.exper.weights.txt")
len_PR_all_seqs = len(PR_all_seq)
with open(PR_weights_path, "r") as f:
    PR_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(PR_weights) == len_PR_all_seqs, "Weights and sequences must have the same length."

PR_pairs = [
    'D30N-N88D',
    'V32I-I47V',
    'G48V-I54A',
    'D30N-K45Q',
    'I54A-V82A',
    'I54V-V82A',
    'M46I-L76V',
    'I54V-V82T',
    'I54A-V82T',
    'G48V-V82A',
    'I54A-A71I',
    'M46I-N88T',
    'L90M-C95F',
    'V32I-M46I',
    'G48V-V82T',
    'I54V-T91S',
    'M46I-F53Y',
    'M46L-K55R',
    'M46L-V82A',
    'M46I-K55R',
    # 'D30N-N88D',
    # 'V32I-I47V',
    # 'K20R-M36I',
    # 'G16E-P39S',
    # 'I54A-V82A',
    # 'I54V-V82A',
    # 'M46I-L76V',
    # 'T12P-K14R',
    # 'L33F-I54L',
    # 'I54V-V82T',
    # 'G73T-L90M',
    # 'G48V-V82A',
    # 'G73S-L90M',
    # 'R57K-Q61N',
    # 'D60E-Q61E',
    # 'M36L-I62V',
    # 'L10I-I54A',
    # 'L10F-I84V',
    # 'P79A-I84V',
    # 'T12S-L19I',
]

if USE_TOP_DDE_PAIRS:
    PR_pairs = load_top_dde_pairs('PR', len_PR_all_seqs)

output_probs('PR', 1,99,PR_all_seq, PR_consensus_seq, PR_redux, PR_pairs, PR_weights_path, PR_J, out_name('protease_all_probabilities_v17', 'PR'))


PR: 24 of 1000 top-ddE pairs have more than 228.4 double-mutation carriers (4.0% of 5710 sequences) -> using 20 (top 20 by ddE)
  gain of fitness: local test only -- contender rule off (NON_OVERLAPPING = False)
Processing pair: D30N-N88D
Processing pair: K20R-M36I
Processing pair: I54V-V82A
Processing pair: G73S-L90M
Processing pair: L10I-I84V
Processing pair: M46L-V82A
Processing pair: I13V-L33F
Processing pair: L10I-G48V
Processing pair: A71V-G73S
Processing pair: E35D-M36I
Processing pair: L24I-V82A
Processing pair: L10I-L24I
Processing pair: A71V-L90M
Processing pair: E35D-N37D
Processing pair: L24I-I54V
Processing pair: L10I-V82A
Processing pair: L33F-I54V
Processing pair: A71V-V82A
Processing pair: K20I-L90M
Processing pair: I54V-A71V
CSV file protease_all_probabilities_v17_topdde_4pct_top20_with_overlap.csv created successfully.


In [6]:
RT_weights_path = 'RT/data/rt.weights.txt'
len_RT_all_seqs = len(RT_all_seq)
RT_weights_path = str(data_root / "RT" / "data" / "rt.weights.txt")
with open(RT_weights_path, "r") as f:
    RT_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the RT_all_seq list
assert len(RT_weights) == len_RT_all_seqs, "Weights and sequences must have the same length."

RT_pairs = [
    'K101E-G190S',
    'K101E-G190A',
    'K103N-P225H',
    'L100I-K103N',
    'K101P-K103S',
    'Y181C-H221Y',
    'K103S-G190A',
    'K103S-P225H',
    'L100I-K103R',
    'V108I-H221Y',
    'K103S-D192N',
    'L100I-K103S',
    'K101E-E138A',
    'Y181C-G190A',
    'K103S-D177N',
    'V108I-V189I',
    'K101E-E138K',
    'E138A-G190E',
    'K101P-D192N',
    'V108I-L109V',

]

# The top-ddE list is not split by drug class, so with the flag on RT is run once,
# as a whole, in the "RT as a whole" cell below.
if USE_TOP_DDE_PAIRS:
    print("USE_TOP_DDE_PAIRS is on -> skipping the NNRTI run; RT is handled as a whole below.")
else:
    output_probs('NNRTI', 39,226,RT_all_seq, RT_consensus_seq, RT_redux, RT_pairs, RT_weights_path, RT_J, 'reverseTranscriptase_NNRTI_probabilities_v17.csv')


USE_TOP_DDE_PAIRS is on -> skipping the NNRTI run; RT is handled as a whole below.


In [7]:
RT_weights_path = str(data_root / "RT" / "data" / "rt.weights.txt")
len_RT_all_seqs = len(RT_all_seq)
with open(RT_weights_path, "r") as f:
    RT_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the RT_all_seq list
assert len(RT_weights) == len_RT_all_seqs, "Weights and sequences must have the same length."

RT_pairs = [
    'F116Y-Q151M',
    'M41L-T215Y',
    'V75I-I132L',
    'K70R-K219E',
    'D67N-K219Q',
    'K70R-K219Q',
    'L210W-T215Y',
    'F116Y-Q151L',
    'K65R-S68N',
    'V75I-F77L',
    'M41L-T215F',
    'D67N-K219E',
    'L210W-T215S',
    'M41L-T215S',
    'D67N-K70R',
    'L74V-Y115F',
    'L74V-L100I',
    'A62V-V75I',
    'F116Y-Q151R',
    'A62V-V75T',
    # 'F116Y-Q151M',
    # 'M41L-T215Y',
    # 'V75M-F77L',
    # 'K101E-G190S',
    # 'E203K-K223E',
    # 'K43E-E44A',
    # 'K70R-K219E',
    # 'D67N-K219Q',
    # 'K70R-K219Q',
    # 'L210W-T215Y',
    # 'K103R-V179D',
    # 'K101E-G190A',
    # 'K103N-P225H',
    # 'L100I-K103N',
    # 'V75I-F77L',
    # 'M41L-T215F',
    # 'D67G-K219E',
    # 'D121Y-K122E',
    # 'D67N-K219E',
    # 'D121H-K122E',
]

# Same as the NNRTI cell: with the flag on, RT is run once as a whole below.
if USE_TOP_DDE_PAIRS:
    print("USE_TOP_DDE_PAIRS is on -> skipping the NRTI run; RT is handled as a whole below.")
else:
    output_probs('NRTI', 39,226,RT_all_seq, RT_consensus_seq, RT_redux, RT_pairs, RT_weights_path, RT_J, 'reverseTranscriptase_NRTI_probabilities_v17.csv')


USE_TOP_DDE_PAIRS is on -> skipping the NRTI run; RT is handled as a whole below.


In [8]:
# ---- RT as a whole (USE_TOP_DDE_PAIRS only) ----------------------------------
# The ms0_5 top-1000 ddE list for RT is one list for the whole protein -- the NRTI and
# NNRTI files hold the same pairs and differ only in which mutations they flag as DRMs --
# so NRTI and NNRTI are pooled here into a single run and a single output file.
RT_weights_path = str(data_root / "RT" / "data" / "rt.weights.txt")
len_RT_all_seqs = len(RT_all_seq)

if USE_TOP_DDE_PAIRS:
    RT_pairs_all = load_top_dde_pairs('RT', len_RT_all_seqs)
    output_probs('both', 39,226,RT_all_seq, RT_consensus_seq, RT_redux, RT_pairs_all, RT_weights_path, RT_J, out_name('reverseTranscriptase_both_probabilities_v17', 'RT'))
else:
    print("USE_TOP_DDE_PAIRS is off -> RT was already run as NRTI / NNRTI in the two cells above.")


RT: 49 of 1000 top-ddE pairs have more than 959.7 double-mutation carriers (5.0% of 19194 sequences) -> using 20 (top 20 by ddE)


  gain of fitness: local test only -- contender rule off (NON_OVERLAPPING = False)
Processing pair: M41L-T215Y
Processing pair: K70R-K219E
Processing pair: D67N-K219Q
Processing pair: K70R-K219Q
Processing pair: L210W-T215Y
Processing pair: M41L-T215F
Processing pair: D67N-K219E
Processing pair: D67N-K70R
Processing pair: V118I-F214L
Processing pair: K122E-D123N
Processing pair: M41L-L210W
Processing pair: D67N-T69D
Processing pair: D67N-T215F
Processing pair: E44D-T215Y
Processing pair: T69N-K70R
Processing pair: M41L-E44D
Processing pair: D67N-D218E
Processing pair: H208Y-R211K
Processing pair: Y181C-G190A
Processing pair: T215F-K219Q
CSV file reverseTranscriptase_both_probabilities_v17_topdde_5pct_top20_with_overlap.csv created successfully.
